In [20]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [21]:
fire_schema = StructType(
    [StructField('CallNumber', IntegerType(), True),
    StructField('UnitID', StringType(), True),
    StructField('IncidentNumber', IntegerType(), True),
    StructField('CallType', StringType(), True),
    StructField('CallDate', StringType(), True),
    StructField('WatchDate', StringType(), True),
    StructField('CallFinalDisposition', StringType(), True),
    StructField('AvailableDtTm', StringType(), True),
    StructField('Address', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Zipcode', IntegerType(), True),
    StructField('Battalion', StringType(), True),
    StructField('StationArea', StringType(), True),
    StructField('Box', StringType(), True),
    StructField('OriginalPriority', StringType(), True),
    StructField('Priority', StringType(), True),
    StructField('FinalPriority', IntegerType(), True),
    StructField('ALSUnit', BooleanType(), True),
    StructField('CallTypeGroup', StringType(), True),
    StructField('NumAlarms', IntegerType(), True),
    StructField('UnitType', StringType(), True),
    StructField('UnitSequenceInCallDispatch', IntegerType(), True),
    StructField('FirePreventionDistrict', StringType(), True),
    StructField('SupervisorDistrict', StringType(), True),
    StructField('Neighborhood', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('RowID', StringType(), True),
    StructField('Delay', FloatType(), True)
    ])

In [22]:
spark = SparkSession.builder.appName("EX-2.3").getOrCreate()

fire_df = spark.read.csv("../data/Fire_Incidents_20260718.csv", header=True, schema=fire_schema)

# Print some few rows and schema
fire_df.show()

+----------+------+--------------+--------------------+----------+---------+--------------------+--------------------+--------------------+-------------+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------+--------------------------+----------------------+------------------+------------+--------+-----+-----+
|CallNumber|UnitID|IncidentNumber|            CallType|  CallDate|WatchDate|CallFinalDisposition|       AvailableDtTm|             Address|         City|Zipcode|Battalion|StationArea| Box|OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|Neighborhood|Location|RowID|Delay|
+----------+------+--------------+--------------------+----------+---------+--------------------+--------------------+--------------------+-------------+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+----

26/07/18 10:02:14 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 66, schema size: 28
CSV file: file:///Users/pius/Projects/analytics-engineering/Spark/data/Fire_Incidents_20260718.csv


In [23]:
fire_df.printSchema()

root
 |-- CallNumber: integer (nullable = true)
 |-- UnitID: string (nullable = true)
 |-- IncidentNumber: integer (nullable = true)
 |-- CallType: string (nullable = true)
 |-- CallDate: string (nullable = true)
 |-- WatchDate: string (nullable = true)
 |-- CallFinalDisposition: string (nullable = true)
 |-- AvailableDtTm: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Battalion: string (nullable = true)
 |-- StationArea: string (nullable = true)
 |-- Box: string (nullable = true)
 |-- OriginalPriority: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- FinalPriority: integer (nullable = true)
 |-- ALSUnit: boolean (nullable = true)
 |-- CallTypeGroup: string (nullable = true)
 |-- NumAlarms: integer (nullable = true)
 |-- UnitType: string (nullable = true)
 |-- UnitSequenceInCallDispatch: integer (nullable = true)
 |-- FirePreventionDistrict: string (nullable = true)
 

In [26]:
# Write the data to a parquet file
fire_df.coalesce(1).write.format('parquet').mode("overwrite").save('../data/fire_incidents_parquet')

26/07/18 10:03:08 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 66, schema size: 28
CSV file: file:///Users/pius/Projects/analytics-engineering/Spark/data/Fire_Incidents_20260718.csv


In [28]:
fire_df.write.format("parquet").saveAsTable('reported_fires')

26/07/18 10:03:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/18 10:03:26 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 66, schema size: 28
CSV file: file:///Users/pius/Projects/analytics-engineering/Spark/data/Fire_Incidents_20260718.csv


### Transformations and actions

#### Projections and filters

A projection in relational parlance is a way to return only the rows matching a certain relational condition by using filters. 

In Spark, projections are done using the `select()` method, while filters can be expressed using the `filter()` or `where()` method.

In [29]:
few_fire_df = (fire_df
    .select("IncidentNumber", "AvailableDtTm", "CallType")
    .where(col("CallType") != "Medical Incident"))

few_fire_df.show(5, truncate=False)

+--------------+----------------------+----------------------+
|IncidentNumber|AvailableDtTm         |CallType              |
+--------------+----------------------+----------------------+
|221330060     |2022/10/15 04:29:50 AM|CLEMENTINA STREET     |
|221330100     |2022/10/15 04:55:58 AM|649 FELL STREET       |
|221330120     |2022/10/15 05:08:00 AM|POLK STREET           |
|221330200     |2022/10/15 05:55:31 AM|2032 MISSION STREET   |
|221330250     |2022/10/15 06:10:32 AM|1 280SB ALEMANY MSN OF|
+--------------+----------------------+----------------------+
only showing top 5 rows


26/07/18 10:03:28 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Address, Arrival DtTm
 Schema: IncidentNumber, CallType, AvailableDtTm
Expected: IncidentNumber but found: ID
CSV file: file:///Users/pius/Projects/analytics-engineering/Spark/data/Fire_Incidents_20260718.csv


In [30]:
# Distinct call types

(fire_df
    .select("CallType")
    .where(col("CallType").isNotNull())
    .agg(count_distinct("CallType").alias("DistinctCallTypes"))
    .show()
    )

26/07/18 10:03:28 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Address
 Schema: CallType
Expected: CallType but found: Address
CSV file: file:///Users/pius/Projects/analytics-engineering/Spark/data/Fire_Incidents_20260718.csv


+-----------------+
|DistinctCallTypes|
+-----------------+
|           178207|
+-----------------+



In [31]:
# Distinct Non-null calltypes

(fire_df
    .select("CallType")
    .where(col("CallType").isNotNull())
    .distinct()
    .show(10, False)
)

26/07/18 10:03:29 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Address
 Schema: CallType
Expected: CallType but found: Address
CSV file: file:///Users/pius/Projects/analytics-engineering/Spark/data/Fire_Incidents_20260718.csv


+------------------------+
|CallType                |
+------------------------+
|24 WILLIE MAYS PZ       |
|KING STREET             |
|806 RED LEAF COURT      |
|1202 SUTTER STREET      |
|381 SAN LEANDRO WAY     |
|1 101SB SAN BRUNO SIL ON|
|1 101NB 3RD ST OF       |
|450 RAYMOND AVENUE      |
|IVY STREET              |
|1101 RHODE ISLAND STREET|
+------------------------+
only showing top 10 rows


In [32]:
# Renaming, adding, and dropping columns

new_fire_df = fire_df.withColumnRenamed(
    "Delay", "ResponseDelayedinMins"
)

(new_fire_df.select("ResponseDelayedinMins")
    .where(col("ResponseDelayedinMins") > 5)
    .show(5, False))

26/07/18 10:03:29 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Number of Alarms
 Schema: Delay
Expected: Delay but found: Number of Alarms
CSV file: file:///Users/pius/Projects/analytics-engineering/Spark/data/Fire_Incidents_20260718.csv


+---------------------+
|ResponseDelayedinMins|
+---------------------+
+---------------------+

